# 💰 Rupee Track — Backend API
Run each cell **top to bottom**. The last cell gives you a public URL — paste it into the Rupee Track app.

**Your data is saved to `transactions.csv` in this Colab session.**  
Mount Google Drive (optional step below) to keep data permanently.

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────
!pip install fastapi uvicorn pyngrok nest-asyncio pandas -q

In [ ]:
# ── CELL 2 (OPTIONAL): Mount Google Drive to save data permanently ─────────
# Skip this cell if you don't need permanent storage.
# If you run it, your transactions.csv will live in your Drive.

from google.colab import drive
drive.mount('/content/drive')
CSV_PATH = '/content/drive/MyDrive/rupeetrack_transactions.csv'
print('Drive mounted. CSV will be saved to:', CSV_PATH)

In [ ]:
# ── CELL 3: Define CSV path (run this even if you skipped Cell 2) ──────────
import os

# If you mounted Drive above, this keeps that path.
# If you skipped Cell 2, data saves locally in Colab (lost when session ends).
if 'CSV_PATH' not in dir():
    CSV_PATH = '/content/transactions.csv'

print('Using CSV path:', CSV_PATH)

In [ ]:
# ── CELL 4: The API server ─────────────────────────────────────────────────
import pandas as pd
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

nest_asyncio.apply()  # lets uvicorn run inside Colab's event loop

app = FastAPI()

# Allow requests from any origin (your GitHub Pages URL, localhost, etc.)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── CSV helpers ────────────────────────────────────────────────────────────
COLUMNS = ['id','date','payment_type','amount','description','category','type']

def load_df():
    """Load CSV into a DataFrame. Creates empty one if file doesn't exist."""
    if os.path.exists(CSV_PATH):
        return pd.read_csv(CSV_PATH)
    return pd.DataFrame(columns=COLUMNS)

def save_df(df):
    """Save DataFrame back to CSV."""
    df.to_csv(CSV_PATH, index=False)

# ── Pydantic model (validates incoming data from the app) ──────────────────
class Transaction(BaseModel):
    date:         str
    payment_type: str
    amount:       float
    description:  str
    category:     str
    type:         str   # 'expense' or 'income'

# ── Endpoints ──────────────────────────────────────────────────────────────

@app.get('/ping')
def ping():
    """Health check — the app pings this to confirm Colab is running."""
    return {'status': 'ok'}


@app.get('/transactions')
def get_transactions():
    """Return all transactions as a list."""
    df = load_df()
    return df.to_dict(orient='records')


@app.post('/transactions')
def create_transaction(data: Transaction):
    """Add a new transaction, save to CSV, return the saved row."""
    df = load_df()
    new_id = int(df['id'].max() + 1) if len(df) else 1
    new_row = {
        'id':           new_id,
        'date':         data.date,
        'payment_type': data.payment_type,
        'amount':       data.amount,
        'description':  data.description,
        'category':     data.category,
        'type':         data.type,
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    save_df(df)
    return new_row


@app.delete('/transactions/{transaction_id}')
def delete_transaction(transaction_id: int):
    """Delete a transaction by id and save."""
    df = load_df()
    if transaction_id not in df['id'].values:
        raise HTTPException(status_code=404, detail='Transaction not found')
    df = df[df['id'] != transaction_id]
    save_df(df)
    return {'message': f'Transaction {transaction_id} deleted'}


@app.get('/analytics')
def get_analytics():
    """
    Data science endpoint — runs Pandas aggregations on your transactions
    and returns structured data for charts.
    This is where your DS project lives.
    """
    df = load_df()
    if df.empty:
        return {'category_totals': [], 'monthly_data': [], 'summary': {}}

    df['date']   = pd.to_datetime(df['date'])
    df['month']  = df['date'].dt.to_period('M').astype(str)
    df['amount'] = df['amount'].astype(float)

    expenses = df[df['type'] == 'expense']
    income   = df[df['type'] == 'income']

    # Category breakdown (pie chart)
    cat_totals = (
        expenses.groupby('category')['amount']
        .sum().reset_index()
        .rename(columns={'amount':'total'})
        .sort_values('total', ascending=False)
        .to_dict(orient='records')
    )

    # Monthly data (bar + line charts)
    monthly_exp = expenses.groupby('month')['amount'].sum().rename('expenses')
    monthly_inc = income.groupby('month')['amount'].sum().rename('income')
    monthly = pd.concat([monthly_inc, monthly_exp], axis=1).fillna(0).reset_index()
    monthly['saved'] = monthly['income'] - monthly['expenses']
    monthly_data = monthly.rename(columns={'month':'label'}).to_dict(orient='records')

    # Summary numbers (stat cards)
    summary = {
        'total_income':      float(income['amount'].sum()),
        'total_expenses':    float(expenses['amount'].sum()),
        'balance':           float(income['amount'].sum() - expenses['amount'].sum()),
        'transaction_count': len(df),
    }

    return {'category_totals': cat_totals, 'monthly_data': monthly_data, 'summary': summary}


print('✅ API defined. Run the next cell to start the server.')

In [ ]:
# ── CELL 5: Start the server with a public URL ─────────────────────────────
# This uses Colab's built-in tunnel — no signup needed.
# It will print a URL like: https://xxxx-xxxx.trycloudflare.com
# Copy that URL and paste it into the Rupee Track app.

from google.colab.output import eval_js

print('Starting server...')
print('When the URL appears below, copy it into the Rupee Track app.')
print('-' * 60)

# Get the public tunnel URL
public_url = eval_js('google.colab.kernel.proxyPort(8000)')
print(f'\n🌐 Your Colab API URL:\n\n   {public_url}\n')
print('Paste this into the app, then click Connect.')
print('-' * 60)
print('Server running... (keep this cell running, do not interrupt)')

uvicorn.run(app, host='0.0.0.0', port=8000)